# A Framework for Leveraging LLMs for Scene Analysis and Cognitive Processing

In [ ]:
!pip install Levenshtein
!pip install nltk       

In [ ]:
import os
HOME = os.getcwd()

os.chdir(HOME)
print(HOME)

In [ ]:
HOME = os.getcwd()
os.chdir(HOME)

DATA_DIR = os.path.join(".", "data", "experiments")
IMAGE_DIR = os.path.join(".", "data", "images")
RESULTS_DIR = os.path.join(".", "outputs")


In [ ]:
from src.models.LLM import LLM, OpenAI, Ollama
from src.models.Framework import SceneUnderstandingFramework
from src.utils.llm_utils import create_llm_instance
from src.core.ImageData import ImageData
from src.models.KG import KnowledgeGraph
import os
import pickle

from src.utils.data_utils import load_image_data

import pandas as pd
pd.set_option('display.max_rows', None)

import pandas as pd  



### Checking the bounding boxes for all participants

In [ ]:
IMG_ID = 1
IMG_TYPE = "exp"
llm_provider = "openai"

# load image
IMG_PATH = os.path.join(IMAGE_DIR, f"{IMG_ID}{IMG_TYPE}.jpg")
image_data = load_image_data(IMG_PATH, IMG_TYPE, target = [430, 510, 545, 595])


# load masks
masks_path_dir = os.path.join(RESULTS_DIR, "Masks", "BestMask")
image_data.load_img_masks( masks_path_dir, part_id=None )
print(f"Loaded a total of {len(image_data.masks)} Masks for IMG {image_data.ID}_{image_data.img_type}")

In [ ]:
image_data.highlight_target();

In [ ]:
image_data.get_scene_masks()

In [ ]:
image_data.plot_mask_labels(  );

### Checking the bounding boxes for 1 participant only

In [ ]:
IMG_ID = 1
IMG_TYPE = "exp"
llm_provider = "openai"

# load image
IMG_PATH = os.path.join(IMAGE_DIR, f"{IMG_ID}{IMG_TYPE}.jpg")
#image_data = load_image_data(IMG_PATH, IMG_TYPE)


# load masks
#masks_path_dir = os.path.join(RESULTS_DIR, "Masks", "BestMask")
#image_data.load_img_masks( masks_path_dir, part_id="2" )        # change here
#print(f"Loaded a total of {len(image_data.masks)} Masks for IMG {image_data.ID}_{image_data.img_type}")

In [ ]:
#image_data.get_scene_masks()

In [ ]:
#image_data.plot_mask_labels(  );

### KG

In [ ]:
# call the KG builder
llm = create_llm_instance("openai", "gpt-4o-mini", temperature = 0.1)
KG = KnowledgeGraph(llm, image_data)

In [ ]:
results = KG.process_image_complete( num_iter=3, debug=False)

In [ ]:

# loading a saved KG
llm = create_llm_instance("openai", "gpt-4o-mini", temperature = 0.1)
KG2 = KnowledgeGraph(llm, image_data)
KG2.load_state(filepath = "outputs\\KnowledgeGraph\\IMG_1_exp_knowledge_graph.pkl")

In [ ]:
KG2.plot_knowledge_graph()

In [ ]:
# this is part of what the KG.process_image_complete( num_iter=3, debug=False) function is doing.

# running this 3x so we can get a more enriched KG
all_triples = []
for i in range(3):

    print(f"Generating triples for iteration {i + 1}...")
    triples_raw = KG.generate_triples(debug=False)  
    triples_cleaned = KG.clean_triples_text(triples_raw)
    lines = triples_cleaned.strip().split('\n')
    print("\nTotal triples generated:", str(len(lines)))

    for line in lines:
        # Remove parentheses and spaces, then split by comma
        triple = tuple(item.strip().replace('(', '').replace(')', '') for item in line.split(','))
        all_triples.append(triple)

print("\nGenerated triples:", str(len(all_triples)))
# Remove duplicates
triples_final = list(set(all_triples))
print("Generated triples after removing duplicates: ", str(len(triples_final)))